In [1]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [ ]:

from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification, TrainingArguments, Trainer, default_data_collator
import evaluate
import numpy as np
from torchvision.transforms import RandomResizedCrop, Compose, Normalize, ToTensor
from PIL import Image
import torch


food = load_dataset("food101", split="train[:5000]")
food_split = food.train_test_split(test_size=0.2)


labels = food_split['train'].features['label'].names
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}


processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
size = (processor.size['height'],processor.size['width'])


transform = Compose([
    RandomResizedCrop(size),
    ToTensor(),
    Normalize(mean=processor.image_mean, std=processor.image_std)
])


def process_image(samples):
    samples['pixel_values'] = [transform(img.convert('RGB')) for img in samples['image']]
    del samples['image']
    return samples

food_split = food_split.with_transform(process_image)


model = AutoModelForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(

    output_dir="./results",
    report_to=[],
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    weight_decay=0.01,
    remove_unused_columns=False,
    save_strategy="epoch",
    learning_rate=0.001,
    num_train_epochs=7,
    logging_steps=10,
    push_to_hub=False,
    metric_for_best_model="accuracy"
)


trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
    train_dataset=food_split['train'],
    eval_dataset=food_split['test'],
    processing_class=processor
)


trainer.train()

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,1.907100,1.932245,0.188000
2,1.818400,1.712781,0.311000
3,1.587100,1.667679,0.332000
4,1.693700,1.616836,0.352000
5,1.537800,1.523275,0.439000
6,1.370700,1.445859,0.474000
7,1.356200,1.353479,0.531000


TrainOutput(global_step=1750, training_loss=1.665116419655936, metrics={'train_runtime': 1432.6771, 'train_samples_per_second': 19.544, 'train_steps_per_second': 1.221, 'total_flos': 2.1717009635328e+18, 'train_loss': 1.665116419655936, 'epoch': 7.0})

In [ ]:
trainer.save_model("/content/drive/MyDrive/model_CV/google")